# YAGO4-19k analogy classification — full run on Colab

Runtime ▸ Change runtime type ▸ **T4 GPU**, then Runtime ▸ Run all (≈ 15–25 min).

In [ ]:
!git clone -q https://github.com/AWARKEH1/yago-analogy-classification.git
%cd yago-analogy-classification
!pip install -q gdown
!nvidia-smi --query-gpu=name --format=csv,noheader

## 1. YAGO4-19k triples
`train.pkl` = training triples of YAGO4-19k (Hubert et al.), shared by the original project on Google Drive.

In [ ]:
!mkdir -p data
!gdown -q 1jvneLm3Jg1_tc2gHqvC4K4sNOI_0AZ2m -O data/train.pkl
!ls -lh data

## 2. Build valid / invalid analogies (split by triples, no leakage)

In [ ]:
!python -m analogy_kg.data --triples data/train.pkl --out data/analogies

## 3. Embed every unique entity once (gte-large-en-v1.5)

In [ ]:
!python -m analogy_kg.embed --analogies data/analogies --out data/embeddings.npz

## 4. Train ANNc and evaluate on the held-out test sets

In [ ]:
!python -m analogy_kg.train --data data/analogies --emb data/embeddings.npz --out results

In [ ]:
from IPython.display import Image, display
for m in ['corrupted', 'cross_relation']:
    display(Image(f'results/figures/training_{m}.png'))

## 5. Try new analogies

In [ ]:
!python -m analogy_kg.predict "Paris" "France" "Rome" "Italy"
!python -m analogy_kg.predict "Paris" "France" "Rome" "Albert Einstein"

## 6. Unit tests

In [ ]:
!pip install -q pytest && python -m pytest -q

## 7. Download the results
Download `results.zip` (metrics + figures) and send it back, to update the README.

In [ ]:
!zip -qr results.zip results/metrics.json results/figures
from google.colab import files
files.download('results.zip')